# 支线 7：BERT — Transformer 的另一半：Encoder-Only 模型

> **背景**：我们一直在讲 GPT（Decoder-Only），但 Transformer 论文里还有一个「Encoder」。
> BERT 就是只用 Encoder 堆叠起来的模型，一度是 NLP 的绝对王者。
>
> 本 Part 目标：**理解 Encoder-Only 架构为什么擅长「理解」而不是「生成」，
> BERT 的输入是怎么构造的，MLM 预训练怎么做，以及它和 GPT 的本质区别。**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

### 1. Encoder vs Decoder：先搞清楚架构上的本质区别

在你脑子里应该有一张图：

```
        Encoder-Only (BERT)          Decoder-Only (GPT)
        ────────────────             ────────────────

        输出: "B-PER"                   输出: "法"
          ↑                               ↑
    ┌─────┴─────┐                   ┌─────┴─────┐
    │  FFN + LN  │                   │  FFN + LN  │
    ├───────────┤                   ├───────────┤
    │ Attention │                   │ Attention │
    │ (双向! )  │                   │ (单向! )  │
    ├───────────┤                   ├───────────┤
    │  Input    │                   │  Input     │
    └───────────┘                   └───────────┘

   每个词能看到所有词              每个词只能看到前面的词
   （包括前面和后面）              （因果掩码）
```

同一个词「苹果」：
- **BERT 看到**：「我今天吃了一个 __ 」 → 前后都看，知道是「名词、食物」
- **GPT 看到**：「我今天吃了一个」 → 只能看前面，任务是猜下一个词

| | Encoder-Only (BERT) | Decoder-Only (GPT) | Encoder-Decoder (T5) |
|------|------|------|------|
| Attention 方向 | **双向**（能看到整句） | **单向**（只看前面） | Encoder 双向 + Decoder 单向 |
| 预训练任务 | **MLM**（遮词填空） | **自回归**（预测下一个） | Span Corruption |
| 擅长 | 理解、分类、标注 | 生成、对话 | 翻译、摘要 |
| 代表模型 | BERT, RoBERTa, DeBERTa | GPT-3/4, LLaMA, Qwen | T5, BART |

### 2. BERT 的输入表示：Token + Segment + Position 三合一

BERT 的输入不是简单的一个 Embedding，而是**三个 Embedding 相加**：

```
输入 = Token Embedding + Segment Embedding + Position Embedding
         ↑                    ↑                    ↑
   "这个词本身是谁"    "属于句子A还是句子B"   "在哪个位置"
```

这和 GPT 的区别在哪？GPT 只有 Token + Position，不需要 Segment（因为 GPT 不处理"两个句子之间的关系"）。

**为什么 BERT 需要 Segment Embedding？** 因为 NSP（下一句预测）任务需要判断两个句子的关系。

下面用代码展示 BERT 的「完整输入构造过程」：

In [ ]:
# ============================================================
# 手动构造 BERT 的输入（从零开始，一步步展示）
# ============================================================

# 模拟一个小 BERT 的配置
VOCAB_SIZE = 100
D_MODEL = 16
MAX_LEN = 20
NUM_SEGMENTS = 2  # 句子 A 或 句子 B

# 三个 Embedding
token_embed = nn.Embedding(VOCAB_SIZE, D_MODEL)
segment_embed = nn.Embedding(NUM_SEGMENTS, D_MODEL)
position_embed = nn.Embedding(MAX_LEN, D_MODEL)

# 模拟输入：两个句子
#   [CLS] 我 喜欢 猫 [SEP] 他 喜欢 狗 [SEP]
#   其中 [CLS]=1, [SEP]=2

sentence_A = [1, 5, 8, 3, 2]    # [CLS] 我 喜欢 猫 [SEP]
sentence_B = [6, 8, 4, 2]        # 他 喜欢 狗 [SEP]
full_ids = sentence_A + sentence_B
seq_len = len(full_ids)

print("Step 1: Token IDs")
print(f"  Input: {full_ids}")
print(f"  含义: [CLS] 我 喜欢 猫 [SEP] 他 喜欢 狗 [SEP]")

# Segment IDs：句子 A 的 token 标 0，句子 B 的 token 标 1
segment_ids = [0] * len(sentence_A) + [1] * len(sentence_B)
print(f"\nStep 2: Segment IDs")
print(f"  Segments: {segment_ids}")
print(f"  含义: 前 {len(sentence_A)} 个属于句子A(0)，后 {len(sentence_B)} 个属于句子B(1)")

# Position IDs：0, 1, 2, ..., seq_len-1
position_ids = list(range(seq_len))
print(f"\nStep 3: Position IDs")
print(f"  Positions: {position_ids}")

# ============================================================
# 三个 Embedding 相加
# ============================================================
tokens_t = torch.tensor(full_ids)
segments_t = torch.tensor(segment_ids)
positions_t = torch.tensor(position_ids)

tok_emb = token_embed(tokens_t)    # (seq_len, D_MODEL)
seg_emb = segment_embed(segments_t)  # (seq_len, D_MODEL)
pos_emb = position_embed(positions_t)  # (seq_len, D_MODEL)

input_embeddings = tok_emb + seg_emb + pos_emb

print(f"\nStep 4: 三个 Embedding 相加")
print(f"  Token Embedding shape:    {tok_emb.shape}")
print(f"  Segment Embedding shape:  {seg_emb.shape}")
print(f"  Position Embedding shape: {pos_emb.shape}")
print(f"  相加后: {input_embeddings.shape}")

# 展示：同一个词（"喜欢"，id=8）在两个句子中的 embedding 不同
# "喜欢" 在句子 A 的位置 2，在句子 B 的位置 1
idx_a = 2  # 句子 A 中的 "喜欢"
idx_b = len(sentence_A) + 1  # 句子 B 中的 "喜欢"

print(f"\n★ 关键证明：同一个词在不同位置 + 不同句子的 embedding 不同")
print(f"  '喜欢' 在句子A (pos={idx_a}, seg=0) 的最终 embedding (前 6 维):")
print(f"    {input_embeddings[idx_a, :6].detach()}")
print(f"  '喜欢' 在句子B (pos={idx_b}, seg=1) 的最终 embedding (前 6 维):")
print(f"    {input_embeddings[idx_b, :6].detach()}")
print(f"  它们一样吗？ {(input_embeddings[idx_a] == input_embeddings[idx_b]).all().item()}")
print(f"  → 即使词相同，但因为位置和 segment 不同，最终 embedding 也不同！")

### 3. BERT 的核心：MLM（遮词填空）预训练

GPT 的训练任务是「给定前面的词，预测下一个词」（自回归）。
BERT 的训练任务是「遮住中间的某些词，让模型猜它们是什么」（MLM）。

```
GPT 的训练:  我 → 爱 → 你 → 中国
            只能看 → 方向

BERT 的训练: 我 [MASK] 你 中国  →  猜 [MASK] 是 "爱"
            看左边   看右边
```

BERT 能同时看到左右两边——因为它的 attention 是**双向**的（没有因果掩码）。
这使它擅长「理解」，但无法生成——因为它训练时从来没学过「一个一个往外蹦词」。

In [ ]:
# ============================================================
# 演示 BERT 的双向 Attention（对比 GPT 的因果 Attention）
# ============================================================
seq_len = 6

# GPT 的因果掩码：每个位置只能看自己和前面的
causal_mask = torch.tril(torch.ones(seq_len, seq_len))

# BERT 没有掩码：每个位置看所有位置（双向）
bert_mask = torch.ones(seq_len, seq_len)

tokens = ["[CLS]", "我", "爱", "你", "中", "[SEP]"]

print("=== GPT 的因果 Attention（单向）===")
print(f"Tokens: {tokens}")
print()
for i in range(seq_len):
    visible = [tokens[j] for j in range(seq_len) if causal_mask[i, j] == 1]
    print(f"  位置 {i} ('{tokens[i]}') 能看到: {visible}")

print(f"\n=== BERT 的 Attention（双向）===")
for i in range(seq_len):
    visible = [tokens[j] for j in range(seq_len) if bert_mask[i, j] == 1]
    print(f"  位置 {i} ('{tokens[i]}') 能看到: {visible}")

print(f"\n关键区别:")
print(f"  GPT:  '你' 看不到 '中国'（还没生成）")
print(f"  BERT: '你' 能看到 '中国'（整句已知，双向看）")
print(f"  → BERT 理解上下文的能力更强！但无法做生成。")

### 4. 完整演示：MLM 是怎么训练的

构建一个 MiniBERT，展示完整的 MLM 训练流程：
1. 随机遮住 15% 的 token
2. 模型预测被遮住的 token
3. Loss 只在被遮住的位置上计算

In [ ]:
# ============================================================
# MiniBERT：只用 Encoder（双向 Attention，无因果掩码）
# ============================================================
class MiniBERTEncoder(nn.Module):
    """和 MiniGPT 类似的 Transformer Block，但没有因果掩码（双向）"""
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, S, D = x.shape
        Q = self.W_Q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # ★ 没有 mask！双向！
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, S, D)
        return self.W_O(out)

class MiniBERTBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attention = MiniBERTEncoder(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class MiniBERT(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2, max_len=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        # BERT 使用学习式位置编码（不是正弦）
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([
            MiniBERTBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.norm_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.max_len = max_len

    def forward(self, x):
        B, S = x.shape
        positions = torch.arange(S, device=x.device).unsqueeze(0)
        x_emb = self.token_embedding(x) + self.position_embedding(positions)
        for block in self.blocks:
            x_emb = block(x_emb)
        x_emb = self.norm_final(x_emb)
        return self.lm_head(x_emb)

print("MiniBERT 定义完毕（双向 Attention，无因果掩码）")

In [ ]:
# ============================================================
# MLM 训练数据构造 + 训练演示
# ============================================================
VOCAB_SIZE = 50
MASK_ID = 3  # [MASK] 的 ID

torch.manual_seed(42)
model = MiniBERT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 模拟训练句子
sentences = [
    [5, 8, 10, 6, 9, 4],   # 我 喜欢 猫 它 很 可爱
    [5, 12, 13, 14, 4],    # 我 讨厌 下雨 天
    [7, 8, 11, 6, 15, 4],  # 他 喜欢 狗 它 忠诚
    [5, 16, 17, 18, 4],    # 我 在 学习 编程
    [7, 16, 19, 20, 4],    # 他 在 听 音乐
    [5, 21, 22, 9, 23, 4], # 我 觉得 数学 很 难
]

# MLM 数据构造函数
def create_mlm_batch(sentences, mask_prob=0.15):
    """
    和真实 BERT 一样：
    - 15% 的 token 被选中
    - 其中 80% 替换为 [MASK]
    - 10% 替换为随机 token
    - 10% 保持不变（但还是要预测）
    """
    # Padding
    max_len = max(len(s) for s in sentences)
    batch_size = len(sentences)
    input_ids = torch.zeros(batch_size, max_len, dtype=torch.long)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)  # -100 = ignore

    for i, sent in enumerate(sentences):
        input_ids[i, :len(sent)] = torch.tensor(sent)

        # 选择要 mask 的位置（15%），但不 mask 第一个和最后一个
        maskable = list(range(1, len(sent) - 1))
        num_mask = max(1, int(len(maskable) * mask_prob))
        mask_positions = torch.randperm(len(maskable))[:num_mask]

        for pos_idx in mask_positions:
            pos = maskable[pos_idx]
            labels[i, pos] = sent[pos]  # 记录原始 token
            rand = torch.rand(1).item()
            if rand < 0.8:
                input_ids[i, pos] = MASK_ID  # 80%: 替换为 [MASK]
            elif rand < 0.9:
                input_ids[i, pos] = torch.randint(5, 25, (1,)).item()  # 10%: 随机
            # 10%: 保持不变

    return input_ids, labels

# 创建 batch
input_ids, labels = create_mlm_batch(sentences)

print("=== MLM 训练数据示例 ===")
print(f"MASK_ID = {MASK_ID}")
print()
for i in range(len(sentences)):
    print(f"句子 {i+1}:")
    print(f"  原始: {sentences[i]}")
    print(f"  输入: {input_ids[i].tolist()}")
    label_row = labels[i].tolist()
    print(f"  labels: {[('MASK→'+str(l) if l != -100 else 'ignore') for l in label_row]}")
    print()

# ============================================================
# 训练
# ============================================================
print("=== 训练 MiniBERT (MLM) ===")
NUM_EPOCHS = 200
losses = []

model.train()
for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    logits = model(input_ids)  # (B, S, V)
    loss = F.cross_entropy(
        logits.view(-1, VOCAB_SIZE),
        labels.view(-1),
        ignore_index=-100  # 忽略非 mask 位置
    )
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if epoch % 20 == 0 or epoch == NUM_EPOCHS - 1:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

print(f"\n初始 Loss: {losses[0]:.4f} → 最终 Loss: {losses[-1]:.4f}")

In [ ]:
# ============================================================
# 测试：MLM 预测效果
# ============================================================
test_sentence = torch.tensor([[5, MASK_ID, 10, 6, MASK_ID, 4]])  # 我 [MASK] 猫 它 [MASK] 可爱

model.eval()
with torch.no_grad():
    logits = model(test_sentence)
    predictions = logits.argmax(dim=-1)

print("=== MLM 预测结果 ===")
print(f"输入:  {test_sentence.tolist()[0]}")
print(f"       ['我', '[MASK]', '猫', '它', '[MASK]', '可爱']")
print(f"预测:  {predictions.tolist()[0]}")
print(f"       ['我', '?', '猫', '它', '?', '可爱']")
print(f"\n位置 1 (预测) = {predictions[0, 1].item()} (期望 8='喜欢')")
print(f"位置 4 (预测) = {predictions[0, 4].item()} (期望 9='很')")

# Visualize loss
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3))
plt.plot(losses, 'b-', linewidth=1)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('MiniBERT MLM Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5. BERT 的微调范式：不同下游任务怎么接？

BERT 预训练完后，针对不同下游任务，只需在顶层加不同的分类头：

| 任务 | 怎么用 BERT | 分类头接在哪里 |
|------|------------|-------------|
| **单句分类**（情感分析） | 输入一句，取 `[CLS]` 的输出 | `[CLS]` 的 hidden → 分类层 |
| **句对分类**（NLI、相似度） | 两句拼一起，`[SEP]` 分隔 | `[CLS]` 的 hidden → 分类层 |
| **序列标注**（NER） | 输入一句 | **每个 token** 的 hidden → 各自的分类层 |
| **问答**（SQuAD） | 问题 + 段落 | 每个 token 的 hidden → 两个分类层（start/end） |

关键洞察：**BERT 本体不动（或微调），只换最后的「头」。**
这和 GPT 的「prompt engineering」是两条路。

In [ ]:
# ============================================================
# 演示：用 MiniBERT 做句子分类
# ============================================================
class MiniBERTForClassification(nn.Module):
    """在 MiniBERT 上接一个分类头"""
    def __init__(self, bert, num_classes):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Linear(bert.token_embedding.embedding_dim, num_classes)

    def forward(self, x):
        hidden = self.bert(x)       # (B, S, D)
        cls_output = hidden[:, 0, :]  # 取 [CLS] 的输出
        return self.classifier(cls_output)  # → (B, num_classes)

# 演示
VOCAB_SIZE = 50
bert_base = MiniBERT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)
clf_model = MiniBERTForClassification(bert_base, num_classes=2)  # 二分类：正面/负面

test_input = torch.randint(0, VOCAB_SIZE, (1, 10))
output = clf_model(test_input)
print(f"输入 shape: {test_input.shape}")
print(f"输出 shape: {output.shape}  ← (batch=1, classes=2)")
print(f"输出 logits: {output.tolist()}")
print(f"\n这就是 BERT 做情感分析的完整流程：")
print(f"  输入句子 → BERT 编码 → 取 [CLS] → 分类层 → 正面/负面")

### 6. 真实 BERT 加载演示（transformers 库）

上面我们实现了 MiniBERT，现在看看真正的 BERT 是怎么用的。

In [ ]:
# ============================================================
# 用 transformers 加载真实 BERT
# ============================================================
try:
    from transformers import AutoTokenizer, AutoModel

    print("加载 BERT-base-chinese...")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
    model = AutoModel.from_pretrained("bert-base-chinese")

    print(f"BERT 参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
    print(f"词表大小: {len(tokenizer)}")
    print()

    # 测试：看 BERT 的 attention 权重
    sentences = [
        "我爱中国",
        "今天天气真好",
    ]

    inputs = tokenizer(sentences, padding=True, return_tensors="pt")
    print(f"输入 IDs shape: {inputs['input_ids'].shape}")
    print(f"Attention mask: {inputs['attention_mask']}")
    print()

    # 前向传播
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    print(f"输出 last_hidden_state shape: {outputs.last_hidden_state.shape}")
    print(f"  → (batch, seq_len, hidden_dim=768)")
    print()

    # 看最后一层的 attention（第一个 head）
    last_layer_attn = outputs.attentions[-1]  # (batch, num_heads, seq_len, seq_len)
    print(f"最后一层 attention shape: {last_layer_attn.shape}")
    print(f"  → (batch, 12 heads, seq_len, seq_len)")
    print()

    # 展示第一个句子每个 token 能看到谁
    tokens1 = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    print(f"句子 1 tokens: {tokens1}")
    print(f"\n每个 token 的 attention 分布（head 0）:")
    for i, tok in enumerate(tokens1):
        attn_weights = last_layer_attn[0, 0, i]  # 第 0 个样本，第 0 个 head，第 i 个 position
        top2 = attn_weights.topk(2)
        print(f"  '{tok}' 最关注: ", end="")
        for j, (idx, w) in enumerate(zip(top2.indices, top2.values)):
            print(f"'{tokens1[idx]}'({w:.2f})", end="  ")
        print()

except ImportError:
    print("transformers 库未安装。运行: pip install transformers")
except Exception as e:
    print(f"加载 BERT 时出错: {e}")
    print("(这是正常的——如果网络不通或模型太大，以上面的 MiniBERT 演示为准)")

### 7. BERT vs GPT：一张表说清本质区别

| | BERT (Encoder-Only) | GPT (Decoder-Only) |
|------|------|------|
| **核心任务** | 理解——「这句话什么意思？」 | 生成——「下一句该说什么？」 |
| **预训练** | MLM（遮词填空，15% mask） | 自回归（预测下一个词） |
| **Attention** | 双向（看整句） | 单向/因果（只看前面） |
| **输入表示** | Token + Segment + Position | Token + Position（无 Segment） |
| **输出** | 每个位置的 hidden state | 每个位置的 logits（但只用最后一个生成） |
| **怎么用** | 加分类头微调 | 改 prompt / 对话格式 |
| **代表模型** | BERT, RoBERTa, DeBERTa | GPT-3/4, LLaMA, Qwen, DeepSeek |
| **为什么 GPT 赢了** | BERT 做不了生成，且规模扩展后涌现能力不如 GPT | GPT 通过 in-context learning 也能做理解任务 |

> BERT 的思想没有消失——MLM 的「遮住一部分让模型猜」策略，被后来的 T5、BART 甚至现代多模态模型吸收。
> - RoBERTa 去掉了 NSP，只用 MLM，更多数据训练更久 → 显著提升
> - DistilBERT 通过知识蒸馏把 BERT 变小 40%，速度快 60%
> - DeBERTa 改进了 attention 机制（解耦注意力），一度超越人类基准

**一句话总结**：BERT 和 GPT 是同一篇 Transformer 论文的两种用法。
BERT 用 Encoder 做「理解」，GPT 用 Decoder 做「生成」。
GPT 赢了规模竞赛，但 BERT 的 MLM 思想和双向 attention 在整个 NLP 领域无处不在。